# Bank Marketing Data Analysis
This notebook presents an analysis of the `bankmarketing.csv` dataset, which includes data related to a bank's marketing campaigns. The main goal is to understand customer behavior and predict whether a client will subscribe to a term deposit.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, roc_auc_score

def clean_and_engineer(file_path):
    df = pd.read_csv(file_path)
    
    df.columns = [col.replace('.', '_') for col in df.columns]
    
    df = df.drop_duplicates()
    
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].replace('unknown', df[col].mode()[0])
    
    df['was_contacted'] = (df['pdays'] != 999).astype(int)
    
    df['age_group'] = pd.cut(df['age'], bins=[0, 25, 40, 60, 100], 
                             labels=['Student/GenZ', 'Young Adult', 'Middle Aged', 'Senior'])
    
    return df

def apply_segmentation(df):
    cluster_cols = ['age', 'campaign', 'emp_var_rate', 'cons_conf_idx']
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df[cluster_cols])
    
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    df['segment_id'] = kmeans.fit_predict(scaled_data)
    
    return df

def build_predictive_model(df):
    X = df.drop(['y', 'age_group'], axis=1)
    y = (df['y'] == 'yes').astype(int)
    
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_features = X.select_dtypes(include=['object']).columns
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])
    
    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=150, max_depth=12, 
                                               random_state=42, class_weight='balanced'))
    ])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                        random_state=42, stratify=y)
    
    model_pipeline.fit(X_train, y_train)
    
    return model_pipeline, X_test, y_test


def generate_visuals(df, model, X_test, y_test):
    plt.style.use('ggplot')
    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    
    all_features = model.named_steps['preprocessor'].get_feature_names_out()
    
    clean_feature_names = [f.split('__')[-1] for f in all_features]
    
    importances = pd.Series(model.named_steps['classifier'].feature_importances_, 
                            index=clean_feature_names).sort_values(ascending=True).tail(10)
    
    importances.plot(kind='barh', ax=axes[0], color='#3498db')
    axes[0].set_title('1. Top 10 Drivers of Subscription', fontsize=15, fontweight='bold')
    axes[0].set_xlabel('Importance Score')
    
    job_conv = df.groupby('job')['y'].apply(lambda x: (x == 'yes').mean()).sort_values(ascending=True)
    job_conv.plot(kind='barh', ax=axes[1], color='#2ecc71')
    axes[1].set_title('2. Conversion Rate by Job Segment', fontsize=15, fontweight='bold')
    axes[1].axvline(df['y'].map({'yes':1, 'no':0}).mean(), color='red', linestyle='--', label='Average')
    axes[1].legend()
    
    sns.kdeplot(data=df[df['duration'] < 1500], x='duration', hue='y', fill=True, ax=axes[2], palette='viridis')
    axes[2].set_title('3. Call Duration Tipping Point', fontsize=15, fontweight='bold')
    axes[2].set_xlim(0, 1500)
    
    plt.tight_layout()
    plt.savefig('bank_analytics_expert_report.png')
    print("Dashboard successfully saved as 'bank_analytics_expert_report.png'")

if __name__ == "__main__":
    DATA_PATH = 'bankmarketing.csv'
    
    print("Step 1: Cleaning data and engineering features...")
    processed_df = clean_and_engineer(DATA_PATH)
    
    print("Step 2: Applying AI-driven customer segmentation...")
    segmented_df = apply_segmentation(processed_df)
    
    print("Step 3: Training predictive model (Random Forest)...")
    trained_model, X_test_set, y_test_set = build_predictive_model(segmented_df)
    
    print("Step 4: Generating executive visualizations...")
    generate_visuals(segmented_df, trained_model, X_test_set, y_test_set)
    
    segmented_df.to_csv('final_bank_strategy_data.csv', index=False)
    print("Step 5: Exported enriched data to 'final_bank_strategy_data.csv'")
    print("\nProcessing Complete.")